# 基于 mDeBERTa-v3-base 的 Fine-tuning 消融实验

本 notebook 用于完成课程作业中的 fine-tuning 阶段实验设计与实现。任务目标是在给定同一条 `prompt` 的条件下，比较 `response_a` 与 `response_b` 两个候选回答，并预测评审者最终更偏好哪一个回答。

根据当前数据集说明，`train.parquet` 中的监督标签 `winner` 仅包含 `model_a` 和 `model_b`，因此本阶段的主任务属于二分类问题。围绕这一设定，本 notebook 进一步设计三组可直接对比的消融实验，以分析不同优化策略对模型性能的影响。

本实验统一采用 `mDeBERTa-v3-base` 作为基础编码模型，并保持相同的输入模板：

- `Prompt: {prompt}`
- `Response A: {response_a}`
- `Response B: {response_b}`

在此基础上，分别实现以下三组实验：

1. `exp1_prop_trunc_cls`：在 baseline 上引入按比例截断策略，使用二分类头进行训练。
2. `exp2_prop_trunc_swap_cls`：在实验 1 的基础上加入 A/B 反转增强，以降低位置偏差对模型学习的影响。
3. `exp3_prop_trunc_swap_reg`：在实验 2 的基础上，将输出层改为单值回归头，预测 `[0, 1]` 区间内的偏好分数，并以 `0.5` 为阈值映射回二分类结果。

该 notebook 为**完整自包含实现**，可直接在 Kaggle Notebook 或本地 Jupyter 环境中顺序执行，无需依赖外部 `.py` 脚本。


## 运行说明

本 notebook 按“配置参数、定义函数、执行训练、汇总结果”的顺序组织，建议严格按照单元顺序执行。

运行前需要重点确认以下配置项：

- `TRAIN_PATH`：训练集 `train.parquet` 的路径。
- `TEST_PATH`：测试集 `test.parquet` 的路径；若暂时不做测试集推理，可保留为空字符串。
- `MODEL_NAME`：预训练模型名称或本地模型目录。
- `SELECTED_EXPERIMENTS`：本次需要运行的实验列表；若只需完成单个实验，可只保留一个实验名。

默认情况下，notebook 会对所选实验执行交叉验证训练，并在 `outputs/<experiment_name>/` 目录下保存以下结果：

- fold 级别的模型权重文件 `checkpoint`
- OOF 预测结果文件
- 交叉验证指标汇总文件
- 若提供 `TEST_PATH`，还会额外导出测试集预测结果与提交文件

评价指标方面，分类实验主要关注 `log_loss`、`accuracy` 与 `macro_f1`；回归实验主要关注 `mse`、`mae`，并额外报告回归分数映射回二分类后的 `accuracy` 与 `log_loss`。


### 手动导入断点续训 checkpoint

如果你是在新的 Kaggle session 里继续训练，可以不用手动复制文件到 `outputs/`。只需要在配置区填写：

- `MANUAL_RESUME_CHECKPOINT_PATH`：上一次输出里的 `*_last.pth`，用于真正断点续训。
- `MANUAL_BEST_CHECKPOINT_PATH`：上一次输出里的 `*.pth`，用于保留之前的 best model；可选，但建议填写。

例如：

```python
MANUAL_RESUME_CHECKPOINT_PATH = "/kaggle/input/your-last-output/outputs/exp2_prop_trunc_swap_cls/exp2_prop_trunc_swap_cls_fold0_last.pth"
MANUAL_BEST_CHECKPOINT_PATH = "/kaggle/input/your-last-output/outputs/exp2_prop_trunc_swap_cls/exp2_prop_trunc_swap_cls_fold0.pth"
```

代码会优先从 `MANUAL_RESUME_CHECKPOINT_PATH` 恢复 optimizer、scheduler、AMP scaler、epoch 与 best score，然后把后续 checkpoint 继续保存到当前 session 的 `outputs/`。


In [ ]:
import math
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
try:
    from sklearn.metrics import accuracy_score, f1_score, log_loss, mean_absolute_error, mean_squared_error
    from sklearn.model_selection import StratifiedKFold
except ImportError:
    def accuracy_score(y_true, y_pred):
        y_true = np.asarray(y_true)
        y_pred = np.asarray(y_pred)
        return float((y_true == y_pred).mean())
    def f1_score(y_true, y_pred, average="macro"):
        if average != "macro":
            raise ValueError("Only macro F1 is supported without scikit-learn")
        y_true = np.asarray(y_true)
        y_pred = np.asarray(y_pred)
        labels = np.unique(np.concatenate([y_true, y_pred]))
        f1_values = []
        for label in labels:
            tp = np.sum((y_true == label) & (y_pred == label))
            fp = np.sum((y_true != label) & (y_pred == label))
            fn = np.sum((y_true == label) & (y_pred != label))
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            if precision + recall == 0:
                f1_values.append(0.0)
            else:
                f1_values.append(2 * precision * recall / (precision + recall))
        return float(np.mean(f1_values))
    def log_loss(y_true, y_pred, labels=None):
        y_true = np.asarray(y_true, dtype=np.int64)
        y_pred = np.asarray(y_pred, dtype=np.float64)
        if y_pred.ndim == 1:
            y_pred = np.stack([1.0 - y_pred, y_pred], axis=1)
        eps = 1e-15
        y_pred = np.clip(y_pred, eps, 1.0 - eps)
        y_pred = y_pred / y_pred.sum(axis=1, keepdims=True)
        if labels is None:
            labels = np.unique(y_true)
        label_to_index = {label: idx for idx, label in enumerate(labels)}
        indices = np.array([label_to_index[label] for label in y_true], dtype=np.int64)
        chosen = y_pred[np.arange(len(y_true)), indices]
        return float(-np.mean(np.log(chosen)))
    def mean_absolute_error(y_true, y_pred):
        y_true = np.asarray(y_true, dtype=np.float64)
        y_pred = np.asarray(y_pred, dtype=np.float64)
        return float(np.mean(np.abs(y_true - y_pred)))
    def mean_squared_error(y_true, y_pred):
        y_true = np.asarray(y_true, dtype=np.float64)
        y_pred = np.asarray(y_pred, dtype=np.float64)
        return float(np.mean((y_true - y_pred) ** 2))
    class StratifiedKFold:
        def __init__(self, n_splits=5, shuffle=False, random_state=None):
            self.n_splits = n_splits
            self.shuffle = shuffle
            self.random_state = random_state
        def split(self, X, y):
            y = np.asarray(y)
            rng = np.random.RandomState(self.random_state)
            fold_indices = [[] for _ in range(self.n_splits)]
            for label in np.unique(y):
                label_indices = np.where(y == label)[0]
                if self.shuffle:
                    rng.shuffle(label_indices)
                label_folds = np.array_split(label_indices, self.n_splits)
                for fold_id, split_indices in enumerate(label_folds):
                    fold_indices[fold_id].extend(split_indices.tolist())
            num_samples = len(y)
            for fold_id in range(self.n_splits):
                valid_idx = np.array(sorted(fold_indices[fold_id]), dtype=np.int64)
                train_mask = np.ones(num_samples, dtype=bool)
                train_mask[valid_idx] = False
                train_idx = np.where(train_mask)[0]
                yield train_idx, valid_idx
TRANSFORMERS_IMPORT_ERROR = None
try:
    from transformers import (
        AutoConfig,
        AutoModelForSequenceClassification,
        AutoTokenizer,
        get_linear_schedule_with_warmup,
    )
except ImportError as exc:
    AutoConfig = None
    AutoModelForSequenceClassification = None
    AutoTokenizer = None
    get_linear_schedule_with_warmup = None
    TRANSFORMERS_IMPORT_ERROR = exc

LABEL2ID = {
    "model_a": 0,
    "model_b": 1,
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}


In [ ]:
@dataclass(frozen=True)
class TrainConfig:
    model_name: str
    max_len: int = 1024
    batch_size: int = 8
    epochs: int = 1
    lr: float = 2e-5
    weight_decay: float = 0.01
    num_folds: int = 5
    seed: int = 42
    num_workers: int = 0
    output_dir: str = "outputs"
    grad_clip: Optional[float] = None
    use_amp: bool = False
    use_data_parallel: bool = False
    head_lr: float = 1e-4
    gradient_accumulation_steps: int = 4
    debug_prediction_stats: bool = True
    pad_to_max_length: bool = False  # False = dynamic padding, saves GPU memory on T4/T4x2
    debug_sample_size: Optional[int] = None  # set e.g. 512 only for quick debugging; None = use all data
    empty_cache_after_fold: bool = True
    resume_training: bool = True
    save_last_checkpoint: bool = True
    skip_finished_folds: bool = True
    manual_resume_checkpoint_path: Optional[str] = "/kaggle/input/models/ai2334135131/model-2/transformers/default/1/exp2_prop_trunc_swap_cls_fold0_last.pth"
    manual_best_checkpoint_path: Optional[str] = None    # manually provide best *.pth path; optional but recommended
    manual_resume_checkpoint_paths: Optional[Dict[str, str]] = None  # optional multi-fold mapping
    manual_best_checkpoint_paths: Optional[Dict[str, str]] = None     # optional multi-fold mapping


@dataclass(frozen=True)
class ExperimentConfig:
    name: str
    task_type: str
    use_proportional_truncation: bool
    use_ab_swap_augmentation: bool
    prompt_ratio: float = 0.2
    response_a_ratio: float = 0.4
    response_b_ratio: float = 0.4
    monitor_metric: str = "accuracy"


EXPERIMENTS: Dict[str, ExperimentConfig] = {
    "exp1_prop_trunc_cls": ExperimentConfig(
        name="exp1_prop_trunc_cls",
        task_type="classification",
        use_proportional_truncation=True,
        use_ab_swap_augmentation=False,
        monitor_metric="accuracy",
    ),
    "exp2_prop_trunc_swap_cls": ExperimentConfig(
        name="exp2_prop_trunc_swap_cls",
        task_type="classification",
        use_proportional_truncation=True,
        use_ab_swap_augmentation=True,
        monitor_metric="accuracy",
    ),
    "exp3_prop_trunc_swap_reg": ExperimentConfig(
        name="exp3_prop_trunc_swap_reg",
        task_type="regression",
        use_proportional_truncation=True,
        use_ab_swap_augmentation=True,
        monitor_metric="accuracy",
    ),
}


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def require_transformers() -> None:
    if TRANSFORMERS_IMPORT_ERROR is not None:
        raise ImportError(
            "transformers is required for tokenizer/model loading. "
            "Install it before running training or inference."
        ) from TRANSFORMERS_IMPORT_ERROR


def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def prepare_runtime(cfg: TrainConfig) -> torch.device:
    device = get_device()
    if device.type == "cuda":
        torch.backends.cudnn.benchmark = True
        gpu_names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
        print(f"Using CUDA with {torch.cuda.device_count()} GPU(s): {gpu_names}")
        print(f"AMP enabled: {cfg.use_amp} | DataParallel enabled: {cfg.use_data_parallel} | head_lr: {cfg.head_lr} | grad_accum: {cfg.gradient_accumulation_steps} | dynamic_padding: {not cfg.pad_to_max_length}")
    else:
        print("CUDA is not available. Falling back to CPU.")
    return device


def maybe_wrap_model(model, device: torch.device, use_data_parallel: bool):
    model = model.to(device)
    if device.type == "cuda" and use_data_parallel and torch.cuda.device_count() > 1:
        print(f"Wrapping model with DataParallel on {torch.cuda.device_count()} GPUs")
        model = torch.nn.DataParallel(model)
    return model


def safe_torch_load(path: Path, map_location=None):
    """Load checkpoints in a way that is compatible with different PyTorch versions."""
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def get_model_state_dict(model):
    if isinstance(model, torch.nn.DataParallel):
        return model.module.state_dict()
    return model.state_dict()


def normalize_state_dict_keys(state_dict):
    """Make checkpoints compatible between single-GPU and DataParallel runs."""
    if not isinstance(state_dict, dict):
        return state_dict
    keys = list(state_dict.keys())
    if keys and all(key.startswith("module.") for key in keys):
        return {key[len("module."):]: value for key, value in state_dict.items()}
    return state_dict


def load_model_state_dict(model, state_dict):
    state_dict = normalize_state_dict_keys(state_dict)
    if isinstance(model, torch.nn.DataParallel):
        model.module.load_state_dict(state_dict)
    else:
        model.load_state_dict(state_dict)


def _clean_optional_path(path_value: Optional[str]) -> Optional[Path]:
    if path_value is None:
        return None
    path_text = str(path_value).strip()
    if not path_text:
        return None
    return Path(path_text)


def _lookup_manual_path(mapping: Optional[Dict[str, str]], exp_cfg: ExperimentConfig, fold: int) -> Optional[Path]:
    """Support several convenient keys for multi-fold manual checkpoint mapping."""
    if not mapping:
        return None
    candidate_keys = [
        f"{exp_cfg.name}_fold{fold}",
        f"{exp_cfg.name}/fold{fold}",
        f"{exp_cfg.name}:{fold}",
        f"fold{fold}",
        str(fold),
        exp_cfg.name,
    ]
    for key in candidate_keys:
        if key in mapping:
            return _clean_optional_path(mapping[key])
    return None


def resolve_manual_resume_path(cfg: TrainConfig, exp_cfg: ExperimentConfig, fold: int) -> Optional[Path]:
    mapped_path = _lookup_manual_path(cfg.manual_resume_checkpoint_paths, exp_cfg, fold)
    if mapped_path is not None:
        return mapped_path
    # A single manual path is intended for the common case: MAX_FOLDS_TO_RUN = 1 and fold0.
    if fold == 0:
        return _clean_optional_path(cfg.manual_resume_checkpoint_path)
    return None


def resolve_manual_best_path(cfg: TrainConfig, exp_cfg: ExperimentConfig, fold: int) -> Optional[Path]:
    mapped_path = _lookup_manual_path(cfg.manual_best_checkpoint_paths, exp_cfg, fold)
    if mapped_path is not None:
        return mapped_path
    # A single manual path is intended for the common case: MAX_FOLDS_TO_RUN = 1 and fold0.
    if fold == 0:
        return _clean_optional_path(cfg.manual_best_checkpoint_path)
    return None


def validate_manual_checkpoint_path(path: Optional[Path], description: str) -> Optional[Path]:
    if path is None:
        return None
    if not path.exists():
        raise FileNotFoundError(f"Manual {description} checkpoint does not exist: {path}")
    return path


def save_best_weight_file_from_checkpoint(src_path: Path, dst_path: Path) -> None:
    """Create a normal best-model .pth file from either a best .pth or a full *_last.pth checkpoint."""
    state = safe_torch_load(src_path, map_location="cpu")
    if isinstance(state, dict) and "model_state_dict" in state:
        state = state["model_state_dict"]
    state = normalize_state_dict_keys(state)
    dst_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(state, dst_path)


def metric_higher_is_better(metric_name: str) -> bool:
    """Return True for metrics where a larger value means a better model."""
    return metric_name in {"accuracy", "macro_f1", "f1", "auc"}


def initial_best_score(metric_name: str) -> float:
    return -float("inf") if metric_higher_is_better(metric_name) else float("inf")


def is_better_score(current_score: float, best_score: float, metric_name: str) -> bool:
    if metric_higher_is_better(metric_name):
        return current_score > best_score
    return current_score < best_score


def save_resume_checkpoint(
    path: Path,
    epoch: int,
    model,
    optimizer,
    scheduler,
    scaler,
    best_score: float,
    best_metrics: Optional[Dict[str, float]],
    cfg: TrainConfig,
    exp_cfg: ExperimentConfig,
    fold: int,
) -> None:
    checkpoint = {
        "epoch": epoch,
        "fold": fold,
        "model_state_dict": get_model_state_dict(model),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
        "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
        "best_score": best_score,
        "best_metrics": best_metrics,
        "best_monitor_metric": exp_cfg.monitor_metric,
        "best_monitor_mode": "max" if metric_higher_is_better(exp_cfg.monitor_metric) else "min",
        "cfg": cfg.__dict__,
        "exp_cfg": exp_cfg.__dict__,
    }
    torch.save(checkpoint, path)


def load_resume_checkpoint(
    path: Path,
    model,
    optimizer,
    scheduler,
    scaler,
    device: torch.device,
    exp_cfg: ExperimentConfig,
) -> Tuple[int, float, Optional[Dict[str, float]]]:
    checkpoint = safe_torch_load(path, map_location=device)
    load_model_state_dict(model, checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    if scheduler is not None and checkpoint.get("scheduler_state_dict") is not None:
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    if scaler is not None and checkpoint.get("scaler_state_dict") is not None:
        scaler.load_state_dict(checkpoint["scaler_state_dict"])

    start_epoch = int(checkpoint.get("epoch", -1)) + 1

    saved_exp_cfg = checkpoint.get("exp_cfg", {}) or {}
    saved_monitor_metric = checkpoint.get("best_monitor_metric", saved_exp_cfg.get("monitor_metric"))

    if saved_monitor_metric != exp_cfg.monitor_metric:
        print(
            f"Resume checkpoint monitor metric changed from {saved_monitor_metric} to {exp_cfg.monitor_metric}; "
            "resetting best_score and best_metrics for the new selection rule."
        )
        best_score = initial_best_score(exp_cfg.monitor_metric)
        best_metrics = None
    else:
        best_score = float(checkpoint.get("best_score", initial_best_score(exp_cfg.monitor_metric)))
        best_metrics = checkpoint.get("best_metrics", None)

    return start_epoch, best_score, best_metrics


def load_checkpoint_model(model_name: str, exp_cfg: ExperimentConfig, checkpoint_path: Path, device: torch.device, use_data_parallel: bool):
    model = get_model(model_name, exp_cfg)
    state = safe_torch_load(checkpoint_path, map_location=device)
    if isinstance(state, dict) and "model_state_dict" in state:
        state = state["model_state_dict"]
    load_model_state_dict(model, state)
    model = maybe_wrap_model(model, device, use_data_parallel)
    return model


def normalize_text(value: object) -> str:
    if pd.isna(value):
        return ""
    return str(value)


def load_data(path: str, is_train: bool = True) -> pd.DataFrame:
    df = pd.read_parquet(path).copy()

    for column in ["prompt", "response_a", "response_b"]:
        if column not in df.columns:
            raise KeyError(f"Missing required column: {column}")
        df[column] = df[column].map(normalize_text)

    if is_train:
        if "winner" not in df.columns:
            raise KeyError("train data must include a winner column")

        df["label_cls"] = df["winner"].map(LABEL2ID)
        if df["label_cls"].isna().any():
            unknown_values = sorted(df.loc[df["label_cls"].isna(), "winner"].dropna().unique())
            raise ValueError(f"Unsupported winner values: {unknown_values}")

        df["label_cls"] = df["label_cls"].astype(int)
        df["label_reg"] = df["label_cls"].astype(np.float32)

    return df


def build_input_text(prompt: str, response_a: str, response_b: str) -> str:
    return (
        f"Prompt: {prompt}\n\n"
        f"Response A: {response_a}\n\n"
        f"Response B: {response_b}"
    )


def allocate_token_budget(lengths: Sequence[int], ratios: Sequence[float], total_budget: int) -> List[int]:
    if total_budget <= 0:
        return [0 for _ in lengths]

    if len(lengths) != len(ratios):
        raise ValueError("lengths and ratios must have the same size")

    ratio_sum = sum(ratios)
    if ratio_sum <= 0:
        raise ValueError("ratios must sum to a positive value")

    raw_targets = [total_budget * ratio / ratio_sum for ratio in ratios]
    base_targets = [int(math.floor(target)) for target in raw_targets]

    remaining_budget = total_budget - sum(base_targets)
    if remaining_budget > 0:
        order = sorted(
            range(len(ratios)),
            key=lambda idx: (raw_targets[idx] - base_targets[idx], ratios[idx]),
            reverse=True,
        )
        for idx in order[:remaining_budget]:
            base_targets[idx] += 1

    allocations = [min(length, target) for length, target in zip(lengths, base_targets)]
    remaining_budget = total_budget - sum(allocations)

    while remaining_budget > 0:
        needy = [idx for idx, (length, alloc) in enumerate(zip(lengths, allocations)) if length > alloc]
        if not needy:
            break

        weight_sum = sum(ratios[idx] for idx in needy)
        growth = [0] * len(lengths)
        used = 0

        if weight_sum > 0:
            raw_growth = {idx: remaining_budget * ratios[idx] / weight_sum for idx in needy}
            base_growth = {idx: int(math.floor(raw_growth[idx])) for idx in needy}

            for idx in needy:
                room = lengths[idx] - allocations[idx]
                inc = min(room, base_growth[idx])
                growth[idx] = inc
                used += inc

            remainder = remaining_budget - used
            if remainder > 0:
                order = sorted(
                    needy,
                    key=lambda idx: (raw_growth[idx] - math.floor(raw_growth[idx]), ratios[idx]),
                    reverse=True,
                )
                for idx in order:
                    if remainder == 0:
                        break
                    room = lengths[idx] - allocations[idx] - growth[idx]
                    if room <= 0:
                        continue
                    growth[idx] += 1
                    used += 1
                    remainder -= 1

        if used == 0:
            for idx in needy:
                allocations[idx] += 1
                remaining_budget -= 1
                if remaining_budget == 0:
                    break
            continue

        for idx, inc in enumerate(growth):
            allocations[idx] += inc
        remaining_budget -= used

    return allocations


def truncate_keep_head_tail_drop_middle(
    token_ids: Sequence[int],
    budget: int,
    is_train: bool,
) -> List[int]:
    token_ids = list(token_ids)

    if budget <= 0:
        return []
    if len(token_ids) <= budget:
        return token_ids
    if budget == 1:
        return [token_ids[0]]
    if budget == 2:
        return [token_ids[0], token_ids[-1]]

    edge_budget = max(1, budget // 4)
    head_keep = min(edge_budget, budget - 1, len(token_ids))
    tail_keep = min(edge_budget, budget - head_keep, len(token_ids) - head_keep)
    middle_budget = budget - head_keep - tail_keep

    head_ids = token_ids[:head_keep]
    tail_ids = token_ids[-tail_keep:] if tail_keep > 0 else []
    middle_ids = token_ids[head_keep: len(token_ids) - tail_keep]

    if middle_budget <= 0 or not middle_ids:
        return (head_ids + tail_ids)[:budget]

    if len(middle_ids) <= middle_budget:
        return head_ids + middle_ids + tail_ids

    drop_count = len(middle_ids) - middle_budget
    max_drop_start = len(middle_ids) - drop_count

    if is_train:
        drop_start = random.randint(0, max_drop_start)
    else:
        drop_start = max_drop_start // 2

    kept_middle = middle_ids[:drop_start] + middle_ids[drop_start + drop_count:]
    return head_ids + kept_middle + tail_ids


def encode_example(
    prompt: str,
    response_a: str,
    response_b: str,
    tokenizer,
    max_len: int,
    ratios: Tuple[float, float, float],
    prompt_prefix_ids: Sequence[int],
    response_a_prefix_ids: Sequence[int],
    response_b_prefix_ids: Sequence[int],
    is_train: bool,
    pad_to_max_length: bool = True,
) -> Dict[str, List[int]]:
    prompt_ids = tokenizer.encode(prompt, add_special_tokens=False)
    response_a_ids = tokenizer.encode(response_a, add_special_tokens=False)
    response_b_ids = tokenizer.encode(response_b, add_special_tokens=False)

    special_tokens = tokenizer.num_special_tokens_to_add(pair=False)
    prefix_tokens = len(prompt_prefix_ids) + len(response_a_prefix_ids) + len(response_b_prefix_ids)
    content_budget = max_len - special_tokens - prefix_tokens

    prompt_budget, response_a_budget, response_b_budget = allocate_token_budget(
        lengths=[len(prompt_ids), len(response_a_ids), len(response_b_ids)],
        ratios=list(ratios),
        total_budget=content_budget,
    )

    prompt_ids = truncate_keep_head_tail_drop_middle(prompt_ids, prompt_budget, is_train=is_train)
    response_a_ids = truncate_keep_head_tail_drop_middle(response_a_ids, response_a_budget, is_train=is_train)
    response_b_ids = truncate_keep_head_tail_drop_middle(response_b_ids, response_b_budget, is_train=is_train)

    body_ids = (
        list(prompt_prefix_ids)
        + prompt_ids
        + list(response_a_prefix_ids)
        + response_a_ids
        + list(response_b_prefix_ids)
        + response_b_ids
    )

    max_body_len = max_len - special_tokens
    if max_body_len > 0:
        body_ids = body_ids[:max_body_len]
    else:
        body_ids = []

    cls_token_id = tokenizer.cls_token_id
    sep_token_id = tokenizer.sep_token_id
    pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0

    if cls_token_id is None or sep_token_id is None:
        raise ValueError("Tokenizer must provide cls_token_id and sep_token_id for proportional truncation")

    max_body_len = max(max_len - 2, 0)
    body_ids = body_ids[:max_body_len]

    input_ids = [cls_token_id] + body_ids + [sep_token_id]
    attention_mask = [1] * len(input_ids)

    encoded = {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "token_type_ids": [0] * len(input_ids),
    }

    for key, values in list(encoded.items()):
        if len(values) > max_len:
            encoded[key] = values[:max_len]

    if pad_to_max_length:
        pad_len = max_len - len(encoded["input_ids"])
        if pad_len > 0:
            encoded["input_ids"] = encoded["input_ids"] + [pad_token_id] * pad_len
            encoded["attention_mask"] = encoded["attention_mask"] + [0] * pad_len
            if "token_type_ids" in encoded:
                encoded["token_type_ids"] = encoded["token_type_ids"] + [0] * pad_len

    return encoded


def augment_swap_ab(train_df: pd.DataFrame, task_type: str) -> pd.DataFrame:
    base_df = train_df.copy()
    base_df["is_swapped"] = False

    swapped_df = train_df.copy()
    swapped_df[["response_a", "response_b"]] = swapped_df[["response_b", "response_a"]].to_numpy()
    swapped_df["label_cls"] = 1 - swapped_df["label_cls"]
    swapped_df["label_reg"] = 1.0 - swapped_df["label_reg"]
    if "winner" in swapped_df.columns:
        swapped_df["winner"] = swapped_df["winner"].map({"model_a": "model_b", "model_b": "model_a"})
    swapped_df["is_swapped"] = True

    augmented_df = pd.concat([base_df, swapped_df], ignore_index=True)
    if task_type == "classification":
        augmented_df["label_cls"] = augmented_df["label_cls"].astype(int)
    else:
        augmented_df["label_reg"] = augmented_df["label_reg"].astype(np.float32)
    return augmented_df


In [ ]:
class ArenaDataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        tokenizer,
        max_len: int,
        exp_cfg: ExperimentConfig,
        is_train: bool,
        pad_to_max_length: bool = True,
    ) -> None:
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.exp_cfg = exp_cfg
        self.is_train = is_train
        self.pad_to_max_length = pad_to_max_length
        self.pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
        self.ratios = (
            exp_cfg.prompt_ratio,
            exp_cfg.response_a_ratio,
            exp_cfg.response_b_ratio,
        )
        self.prompt_prefix_ids = tokenizer.encode("Prompt: ", add_special_tokens=False)
        self.response_a_prefix_ids = tokenizer.encode("\n\nResponse A: ", add_special_tokens=False)
        self.response_b_prefix_ids = tokenizer.encode("\n\nResponse B: ", add_special_tokens=False)

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        row = self.df.iloc[idx]

        if self.exp_cfg.use_proportional_truncation:
            encoded = encode_example(
                prompt=row["prompt"],
                response_a=row["response_a"],
                response_b=row["response_b"],
                tokenizer=self.tokenizer,
                max_len=self.max_len,
                ratios=self.ratios,
                prompt_prefix_ids=self.prompt_prefix_ids,
                response_a_prefix_ids=self.response_a_prefix_ids,
                response_b_prefix_ids=self.response_b_prefix_ids,
                is_train=self.is_train,
                pad_to_max_length=self.pad_to_max_length,
            )
        else:
            encoded = self.tokenizer(
                build_input_text(row["prompt"], row["response_a"], row["response_b"]),
                truncation=True,
                max_length=self.max_len,
                padding="max_length" if self.pad_to_max_length else False,
            )

        item = {
            "input_ids": torch.tensor(encoded["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(encoded["attention_mask"], dtype=torch.long),
        }

        if "token_type_ids" in encoded:
            item["token_type_ids"] = torch.tensor(encoded["token_type_ids"], dtype=torch.long)

        if self.is_train:
            if self.exp_cfg.task_type == "classification":
                item["labels"] = torch.tensor(row["label_cls"], dtype=torch.long)
            else:
                item["labels"] = torch.tensor(row["label_reg"], dtype=torch.float32)

        return item



    def collate_fn(self, batch: List[Dict[str, torch.Tensor]]) -> Dict[str, torch.Tensor]:
        """Dynamically pad each mini-batch to its own longest sequence.

        This greatly reduces memory compared with padding every sample to cfg.max_len,
        especially when max_len is 768/1024 on T4 GPUs.
        """
        if self.pad_to_max_length:
            return {key: torch.stack([item[key] for item in batch]) for key in batch[0].keys()}

        output: Dict[str, torch.Tensor] = {}
        sequence_keys = [key for key in ["input_ids", "attention_mask", "token_type_ids"] if key in batch[0]]
        max_batch_len = max(item["input_ids"].numel() for item in batch)

        for key in sequence_keys:
            pad_value = self.pad_token_id if key == "input_ids" else 0
            dtype = batch[0][key].dtype
            padded = torch.full((len(batch), max_batch_len), pad_value, dtype=dtype)
            for i, item in enumerate(batch):
                values = item[key]
                padded[i, : values.numel()] = values
            output[key] = padded

        if "labels" in batch[0]:
            output["labels"] = torch.stack([item["labels"] for item in batch])
        return output


def get_tokenizer(model_name: str):
    require_transformers()
    try:
        return AutoTokenizer.from_pretrained(
            model_name,
            use_fast=False,
            fix_mistral_regex=True,
        )
    except TypeError:
        return AutoTokenizer.from_pretrained(model_name, use_fast=False)


def get_model(model_name: str, exp_cfg: ExperimentConfig):
    require_transformers()
    config = AutoConfig.from_pretrained(model_name)
    config.num_labels = 2 if exp_cfg.task_type == "classification" else 1
    model = AutoModelForSequenceClassification.from_pretrained(model_name, config=config)
    model = model.float()
    return model


def create_loader(
    dataset: Dataset,
    batch_size: int,
    shuffle: bool,
    num_workers: int,
) -> DataLoader:
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=(num_workers > 0),
        collate_fn=getattr(dataset, "collate_fn", None),
    )


def forward_model(model, batch: Dict[str, torch.Tensor], device: torch.device):
    model_inputs = {
        "input_ids": batch["input_ids"].to(device),
        "attention_mask": batch["attention_mask"].to(device),
    }
    if "token_type_ids" in batch:
        model_inputs["token_type_ids"] = batch["token_type_ids"].to(device)
    return model(**model_inputs)


def compute_loss(logits: torch.Tensor, labels: torch.Tensor, exp_cfg: ExperimentConfig,weight: Optional[torch.Tensor] = None,) -> torch.Tensor:
    logits = logits.float()
    if exp_cfg.task_type == "classification":
        return F.cross_entropy(logits, labels.long(), weight=weight)

    scores = torch.sigmoid(logits.squeeze(-1))
    return F.mse_loss(scores, labels.float())


def train_one_epoch(
    model,
    loader: DataLoader,
    optimizer,
    scheduler,
    device: torch.device,
    exp_cfg: ExperimentConfig,
    grad_clip: Optional[float] = None,
    scaler=None,
    use_amp: bool = False,
    gradient_accumulation_steps: int = 1,
    weight: Optional[torch.Tensor] = None,
) -> float:
    model.train()
    total_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(tqdm(loader, total=len(loader), desc="train", leave=False)):
        labels = batch["labels"].to(device)

        with torch.amp.autocast(device_type="cuda", enabled=(use_amp and device.type == "cuda")):
            outputs = forward_model(model, batch, device)
            loss = compute_loss(outputs.logits, labels, exp_cfg,weight=weight)
            loss_for_backward = loss / gradient_accumulation_steps

        if scaler is not None and scaler.is_enabled():
            scaler.scale(loss_for_backward).backward()
        else:
            loss_for_backward.backward()

        should_step = ((step + 1) % gradient_accumulation_steps == 0) or ((step + 1) == len(loader))
        if should_step:
            if scaler is not None and scaler.is_enabled():
                if grad_clip is not None:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                scaler.step(optimizer)
                scaler.update()
            else:
                if grad_clip is not None:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()

            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        total_loss += loss.item()

    return total_loss / max(len(loader), 1)


@torch.no_grad()
def valid_one_epoch(
    model,
    loader: DataLoader,
    device: torch.device,
    exp_cfg: ExperimentConfig,
    use_amp: bool = False,
    weight: Optional[torch.Tensor] = None,
):
    model.eval()
    total_loss = 0.0

    all_labels: List[np.ndarray] = []
    all_probs: List[np.ndarray] = []
    all_scores: List[np.ndarray] = []

    for batch in tqdm(loader, total=len(loader), desc="valid", leave=False):
        labels = batch["labels"].to(device)
        with torch.amp.autocast(device_type="cuda", enabled=(use_amp and device.type == "cuda")):
            outputs = forward_model(model, batch, device)
            loss = compute_loss(outputs.logits, labels, exp_cfg,weight=weight)
        total_loss += loss.item()

        if exp_cfg.task_type == "classification":
            probs = torch.softmax(outputs.logits.float(), dim=-1)
            all_probs.append(probs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
        else:
            scores = torch.sigmoid(outputs.logits.float().squeeze(-1))
            all_scores.append(scores.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    labels_np = np.concatenate(all_labels).astype(np.int64 if exp_cfg.task_type == "classification" else np.float32)
    metrics = {"valid_loss": total_loss / max(len(loader), 1)}

    if exp_cfg.task_type == "classification":
        probs_np = np.concatenate(all_probs)
        preds_np = probs_np.argmax(axis=1)
        metrics.update(
            {
                "log_loss": log_loss(labels_np, probs_np, labels=[0, 1]),
                "accuracy": accuracy_score(labels_np, preds_np),
                "macro_f1": f1_score(labels_np, preds_np, average="macro"),
            }
        )
        return metrics, {"labels": labels_np, "probs": probs_np, "preds": preds_np}

    scores_np = np.concatenate(all_scores).astype(np.float32)
    probs_np = np.stack([1.0 - scores_np, scores_np], axis=1)
    preds_np = (scores_np >= 0.5).astype(np.int64)
    cls_labels = labels_np.astype(np.int64)
    metrics.update(
        {
            "mse": mean_squared_error(labels_np, scores_np),
            "mae": mean_absolute_error(labels_np, scores_np),
            "accuracy": accuracy_score(cls_labels, preds_np),
            "log_loss": log_loss(cls_labels, probs_np, labels=[0, 1]),
        }
    )
    return metrics, {"labels": labels_np, "scores": scores_np, "preds": preds_np, "probs": probs_np}


@torch.no_grad()
def predict_one_epoch(
    model,
    loader: DataLoader,
    device: torch.device,
    exp_cfg: ExperimentConfig,
    use_amp: bool = False,
):
    model.eval()

    all_probs: List[np.ndarray] = []
    all_scores: List[np.ndarray] = []

    for batch in tqdm(loader, total=len(loader), desc="predict", leave=False):
        with torch.amp.autocast(device_type="cuda", enabled=(use_amp and device.type == "cuda")):
            outputs = forward_model(model, batch, device)

        if exp_cfg.task_type == "classification":
            probs = torch.softmax(outputs.logits.float(), dim=-1)
            all_probs.append(probs.cpu().numpy())
        else:
            scores = torch.sigmoid(outputs.logits.float().squeeze(-1))
            all_scores.append(scores.cpu().numpy())

    if exp_cfg.task_type == "classification":
        probs_np = np.concatenate(all_probs)
        preds_np = probs_np.argmax(axis=1)
        return {"probs": probs_np, "preds": preds_np}

    scores_np = np.concatenate(all_scores).astype(np.float32)
    probs_np = np.stack([1.0 - scores_np, scores_np], axis=1)
    preds_np = (scores_np >= 0.5).astype(np.int64)
    return {"scores": scores_np, "probs": probs_np, "preds": preds_np}


In [ ]:
def metrics_to_string(metrics: Dict[str, float]) -> str:
    parts = []
    for key, value in metrics.items():
        parts.append(f"{key}: {value:.4f}")
    return " | ".join(parts)


def build_oof_frame(valid_df: pd.DataFrame, fold: int, outputs: Dict[str, np.ndarray], exp_cfg: ExperimentConfig) -> pd.DataFrame:
    frame = valid_df[["id"]].copy()
    if "language" in valid_df.columns:
        frame["language"] = valid_df["language"].values
    frame["fold"] = fold
    frame["true_label"] = valid_df["label_cls"].values

    if exp_cfg.task_type == "classification":
        probs = outputs["probs"]
        frame["prob_model_a"] = probs[:, 0]
        frame["prob_model_b"] = probs[:, 1]
        frame["pred_label"] = outputs["preds"]
    else:
        probs = outputs["probs"]
        frame["prob_model_a"] = probs[:, 0]
        frame["prob_model_b"] = probs[:, 1]
        frame["score"] = outputs["scores"]
        frame["pred_label"] = outputs["preds"]

    frame["true_winner"] = frame["true_label"].map(ID2LABEL)
    frame["pred_winner"] = frame["pred_label"].map(ID2LABEL)
    return frame


def summarize_metrics(fold_metrics: List[Dict[str, float]]) -> pd.DataFrame:
    metric_df = pd.DataFrame(fold_metrics)
    mean_row = {"fold": "mean"}
    std_row = {"fold": "std"}
    numeric_cols = [col for col in metric_df.columns if col != "fold"]

    for col in numeric_cols:
        mean_row[col] = metric_df[col].mean()
        std_row[col] = metric_df[col].std(ddof=0)

    metric_df = pd.concat([metric_df, pd.DataFrame([mean_row, std_row])], ignore_index=True)
    return metric_df


def save_submission(
    test_df: pd.DataFrame,
    outputs: Dict[str, np.ndarray],
    exp_cfg: ExperimentConfig,
    save_dir: Path,
) -> None:
    pred_labels = outputs["preds"]
    submission = pd.DataFrame(
        {
            "id": test_df["id"].values,
            "winner": pd.Series(pred_labels).map(ID2LABEL).values,
        }
    )
    submission.to_csv(save_dir / f"{exp_cfg.name}_submission.csv", index=False)

    analysis = test_df[["id"]].copy()
    if exp_cfg.task_type == "classification":
        probs = outputs["probs"]
        analysis["prob_model_a"] = probs[:, 0]
        analysis["prob_model_b"] = probs[:, 1]
    else:
        probs = outputs["probs"]
        analysis["prob_model_a"] = probs[:, 0]
        analysis["prob_model_b"] = probs[:, 1]
        analysis["score"] = outputs["scores"]
    analysis["pred_label"] = pred_labels
    analysis["pred_winner"] = pd.Series(pred_labels).map(ID2LABEL).values
    analysis.to_csv(save_dir / f"{exp_cfg.name}_test_predictions.csv", index=False)


def build_optimizer(model, cfg: TrainConfig):
    no_decay = ["bias", "LayerNorm.bias", "LayerNorm.weight"]
    head_keywords = ["classifier", "pooler"]
    backbone_decay, backbone_no_decay = [], []
    head_decay, head_no_decay = [], []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        is_head = any(keyword in name for keyword in head_keywords)
        use_no_decay = any(token in name for token in no_decay)
        if is_head and use_no_decay:
            head_no_decay.append(param)
        elif is_head:
            head_decay.append(param)
        elif use_no_decay:
            backbone_no_decay.append(param)
        else:
            backbone_decay.append(param)

    param_groups = [
        {"params": backbone_decay, "lr": cfg.lr, "weight_decay": cfg.weight_decay},
        {"params": backbone_no_decay, "lr": cfg.lr, "weight_decay": 0.0},
        {"params": head_decay, "lr": cfg.head_lr, "weight_decay": cfg.weight_decay},
        {"params": head_no_decay, "lr": cfg.head_lr, "weight_decay": 0.0},
    ]
    param_groups = [group for group in param_groups if group["params"]]
    return torch.optim.AdamW(param_groups)


def print_prediction_diagnostics(outputs: Dict[str, np.ndarray], exp_cfg: ExperimentConfig, prefix: str = ""):
    if exp_cfg.task_type == "classification":
        prob_b = outputs["probs"][:, 1]
        pred_counts = pd.Series(outputs["preds"]).value_counts().to_dict()
        print(f"{prefix}prob_model_b mean={prob_b.mean():.6f} std={prob_b.std():.6f} min={prob_b.min():.6f} max={prob_b.max():.6f}")
        print(f"{prefix}pred_label counts={pred_counts}")
    else:
        scores = outputs["scores"]
        pred_counts = pd.Series(outputs["preds"]).value_counts().to_dict()
        print(f"{prefix}score mean={scores.mean():.6f} std={scores.std():.6f} min={scores.min():.6f} max={scores.max():.6f}")
        print(f"{prefix}pred_label counts={pred_counts}")


def run_cv(train_path: str, cfg: TrainConfig, exp_cfg: ExperimentConfig, weight: Optional[torch.Tensor] = None):
    set_seed(cfg.seed)
    require_transformers()
    device = prepare_runtime(cfg)
    tokenizer = get_tokenizer(cfg.model_name)
    df = load_data(train_path, is_train=True)
    if cfg.debug_sample_size is not None:
        sample_n = min(int(cfg.debug_sample_size), len(df))
        print(f"Debug mode: sampling {sample_n}/{len(df)} training rows")
        df = df.sample(n=sample_n, random_state=cfg.seed).reset_index(drop=True)

    exp_dir = Path(cfg.output_dir) / exp_cfg.name
    exp_dir.mkdir(parents=True, exist_ok=True)

    skf = StratifiedKFold(n_splits=cfg.num_folds, shuffle=True, random_state=cfg.seed)

    fold_metrics: List[Dict[str, float]] = []
    oof_frames: List[pd.DataFrame] = []
    checkpoint_paths: List[Path] = []

    for fold, (train_idx, valid_idx) in enumerate(skf.split(df, df["label_cls"])):
        if fold >= MAX_FOLDS_TO_RUN:
            break
        print(f"\n========== {exp_cfg.name} | Fold {fold} ==========")

        checkpoint_path = exp_dir / f"{exp_cfg.name}_fold{fold}.pth"          # best model only, used for inference
        resume_path = exp_dir / f"{exp_cfg.name}_fold{fold}_last.pth"         # full training state, used for resume
        manual_resume_path = validate_manual_checkpoint_path(
            resolve_manual_resume_path(cfg, exp_cfg, fold),
            description=f"resume fold{fold}",
        )
        manual_best_path = validate_manual_checkpoint_path(
            resolve_manual_best_path(cfg, exp_cfg, fold),
            description=f"best fold{fold}",
        )
        active_resume_path = manual_resume_path if manual_resume_path is not None else resume_path

        if manual_resume_path is not None:
            print(f"Manual resume checkpoint for fold {fold}: {manual_resume_path}")
        if manual_best_path is not None:
            print(f"Manual best checkpoint for fold {fold}: {manual_best_path}")
            if not checkpoint_path.exists():
                save_best_weight_file_from_checkpoint(manual_best_path, checkpoint_path)
                print(f"Copied manual best weights to current output path: {checkpoint_path}")
        elif manual_resume_path is not None and not checkpoint_path.exists():
            # Fallback: if only *_last.pth is provided, use its model weights as the initial best file.
            # This is enough to continue and produce inference outputs, but providing MANUAL_BEST_CHECKPOINT_PATH is better.
            save_best_weight_file_from_checkpoint(manual_resume_path, checkpoint_path)
            print(f"No manual best checkpoint provided; initialized best weights from resume checkpoint: {checkpoint_path}")

        fold_oof_path = exp_dir / f"{exp_cfg.name}_fold{fold}_oof.csv"
        fold_metric_path = exp_dir / f"{exp_cfg.name}_fold{fold}_metrics.csv"

        if (
            cfg.resume_training
            and cfg.skip_finished_folds
            and manual_resume_path is None
            and checkpoint_path.exists()
            and fold_oof_path.exists()
            and fold_metric_path.exists()
        ):
            print(f"Fold {fold} already finished. Loading saved fold outputs and skipping training.")
            fold_metric_row = pd.read_csv(fold_metric_path).iloc[0].to_dict()
            if "fold" in fold_metric_row:
                fold_metric_row["fold"] = int(fold_metric_row["fold"])
            fold_metrics.append(fold_metric_row)
            oof_frames.append(pd.read_csv(fold_oof_path))
            checkpoint_paths.append(checkpoint_path)
            continue

        train_df = df.iloc[train_idx].reset_index(drop=True)
        valid_df = df.iloc[valid_idx].reset_index(drop=True)

        # 增加类别权重
        class_counts = train_df["label_cls"].value_counts().sort_index()
        class_weights = len(train_df) / (2.0 * class_counts.values)
        class_weights = torch.tensor(class_weights, dtype=torch.float32, device=device)

        if exp_cfg.use_ab_swap_augmentation:
            train_df = augment_swap_ab(train_df, exp_cfg.task_type)

        train_dataset = ArenaDataset(train_df, tokenizer, cfg.max_len, exp_cfg, is_train=True, pad_to_max_length=cfg.pad_to_max_length)
        valid_dataset = ArenaDataset(valid_df, tokenizer, cfg.max_len, exp_cfg, is_train=True, pad_to_max_length=cfg.pad_to_max_length)

        train_loader = create_loader(train_dataset, cfg.batch_size, True, cfg.num_workers)
        valid_loader = create_loader(valid_dataset, cfg.batch_size, False, cfg.num_workers)

        model = maybe_wrap_model(get_model(cfg.model_name, exp_cfg), device, cfg.use_data_parallel)
        optimizer = build_optimizer(model, cfg)
        amp_enabled = cfg.use_amp and device.type == "cuda" and next(model.parameters()).dtype == torch.float32
        try:
            scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)
        except TypeError:
            scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)

        effective_optimizer_steps = max((len(train_loader) + cfg.gradient_accumulation_steps - 1) // cfg.gradient_accumulation_steps, 1)
        num_train_steps = effective_optimizer_steps * cfg.epochs
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=int(0.1 * num_train_steps),
            num_training_steps=num_train_steps,
        )

        best_score = initial_best_score(exp_cfg.monitor_metric)
        best_metrics: Optional[Dict[str, float]] = None
        best_outputs = None
        start_epoch = 0

        if cfg.resume_training and active_resume_path.exists():
            start_epoch, best_score, best_metrics = load_resume_checkpoint(
                active_resume_path,
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                scaler=scaler,
                device=device,
                exp_cfg=exp_cfg,
            )
            if start_epoch >= cfg.epochs:
                print(f"Resume checkpoint already reached epoch {start_epoch}/{cfg.epochs}. Rebuilding fold outputs from best checkpoint.")
            else:
                print(
                    f"Resumed fold {fold} from {active_resume_path} | "
                    f"next epoch: {start_epoch + 1}/{cfg.epochs} | "
                    f"best_{exp_cfg.monitor_metric}: {best_score:.6f}"
                )

        for epoch in range(start_epoch, cfg.epochs):
            print(f"--- Epoch {epoch + 1}/{cfg.epochs} ---")
            train_loss = train_one_epoch(
                model=model,
                loader=train_loader,
                optimizer=optimizer,
                scheduler=scheduler,
                device=device,
                exp_cfg=exp_cfg,
                grad_clip=cfg.grad_clip,
                scaler=scaler,
                use_amp=amp_enabled,
                gradient_accumulation_steps=cfg.gradient_accumulation_steps,
                weight=class_weights,
            )
            metrics, outputs = valid_one_epoch(
                model,
                valid_loader,
                device,
                exp_cfg,
                use_amp=amp_enabled,
                weight=class_weights,
            )
            metrics["train_loss"] = train_loss

            print(metrics_to_string(metrics))
            if cfg.debug_prediction_stats:
                print_prediction_diagnostics(outputs, exp_cfg, prefix="    ")

            current_score = metrics[exp_cfg.monitor_metric]
            if is_better_score(current_score, best_score, exp_cfg.monitor_metric):
                best_score = current_score
                best_metrics = metrics.copy()
                best_outputs = outputs
                torch.save(get_model_state_dict(model), checkpoint_path)
                print(f"    New best checkpoint saved to: {checkpoint_path} | best_{exp_cfg.monitor_metric}: {best_score:.6f}")

            if cfg.save_last_checkpoint:
                save_resume_checkpoint(
                    resume_path,
                    epoch=epoch,
                    model=model,
                    optimizer=optimizer,
                    scheduler=scheduler,
                    scaler=scaler,
                    best_score=best_score,
                    best_metrics=best_metrics,
                    cfg=cfg,
                    exp_cfg=exp_cfg,
                    fold=fold,
                )
                print(f"    Last checkpoint saved to: {resume_path}")

        if not checkpoint_path.exists():
            raise RuntimeError(f"Training did not produce a best checkpoint for fold {fold}")

        # 为了保证中断续训后也能生成 OOF，这里统一加载 best model 再跑一次验证集。
        best_state = safe_torch_load(checkpoint_path, map_location=device)
        if isinstance(best_state, dict) and "model_state_dict" in best_state:
            best_state = best_state["model_state_dict"]
        load_model_state_dict(model, best_state)

        eval_metrics, best_outputs = valid_one_epoch(
            model,
            valid_loader,
            device,
            exp_cfg,
            use_amp=amp_enabled,
            weight=class_weights,
        )

        # 保留 best epoch 当时记录的 train_loss；验证指标用重新评估的 best model 结果。
        if best_metrics is not None and "train_loss" in best_metrics:
            eval_metrics["train_loss"] = best_metrics["train_loss"]
        best_metrics = eval_metrics

        fold_metric_row = {"fold": fold, **best_metrics}
        fold_oof = build_oof_frame(valid_df, fold, best_outputs, exp_cfg)

        pd.DataFrame([fold_metric_row]).to_csv(fold_metric_path, index=False)
        fold_oof.to_csv(fold_oof_path, index=False)

        fold_metrics.append(fold_metric_row)
        checkpoint_paths.append(checkpoint_path)
        oof_frames.append(fold_oof)

        if cfg.empty_cache_after_fold:
            del model, optimizer, scheduler, train_loader, valid_loader, train_dataset, valid_dataset
            if device.type == "cuda":
                torch.cuda.empty_cache()

    if not oof_frames:
        raise RuntimeError("No folds were run or loaded. Check MAX_FOLDS_TO_RUN and saved fold files.")

    oof_df = pd.concat(oof_frames, ignore_index=True)
    metric_df = summarize_metrics(fold_metrics)

    oof_df.to_csv(exp_dir / f"{exp_cfg.name}_oof.csv", index=False)
    metric_df.to_csv(exp_dir / f"{exp_cfg.name}_cv_metrics.csv", index=False)

    print(f"\n========== {exp_cfg.name} | CV Summary ==========")
    print(metric_df.to_string(index=False))

    return checkpoint_paths, oof_df, metric_df



def predict_test(
    test_path: str,
    cfg: TrainConfig,
    exp_cfg: ExperimentConfig,
    checkpoint_paths: Sequence[Path],
) -> None:
    if not test_path:
        return

    require_transformers()
    device = prepare_runtime(cfg)
    tokenizer = get_tokenizer(cfg.model_name)
    test_df = load_data(test_path, is_train=False)
    test_dataset = ArenaDataset(test_df, tokenizer, cfg.max_len, exp_cfg, is_train=False, pad_to_max_length=cfg.pad_to_max_length)
    test_loader = create_loader(test_dataset, cfg.batch_size, False, cfg.num_workers)

    exp_dir = Path(cfg.output_dir) / exp_cfg.name
    exp_dir.mkdir(parents=True, exist_ok=True)

    if exp_cfg.task_type == "classification":
        prob_sum = None
        for checkpoint_path in checkpoint_paths:
            model = load_checkpoint_model(cfg.model_name, exp_cfg, checkpoint_path, device, cfg.use_data_parallel)
            fold_outputs = predict_one_epoch(model, test_loader, device, exp_cfg, use_amp=(cfg.use_amp and next(model.parameters()).dtype == torch.float32))
            if prob_sum is None:
                prob_sum = fold_outputs["probs"]
            else:
                prob_sum += fold_outputs["probs"]

        avg_probs = prob_sum / len(checkpoint_paths)
        avg_preds = avg_probs.argmax(axis=1)
        save_submission(test_df, {"probs": avg_probs, "preds": avg_preds}, exp_cfg, exp_dir)
        return

    score_sum = None
    for checkpoint_path in checkpoint_paths:
        model = load_checkpoint_model(cfg.model_name, exp_cfg, checkpoint_path, device, cfg.use_data_parallel)
        fold_outputs = predict_one_epoch(model, test_loader, device, exp_cfg, use_amp=cfg.use_amp)
        if score_sum is None:
            score_sum = fold_outputs["scores"]
        else:
            score_sum += fold_outputs["scores"]

    avg_scores = score_sum / len(checkpoint_paths)
    avg_probs = np.stack([1.0 - avg_scores, avg_scores], axis=1)
    avg_preds = (avg_scores >= 0.5).astype(np.int64)
    save_submission(
        test_df,
        {"scores": avg_scores, "probs": avg_probs, "preds": avg_preds},
        exp_cfg,
        exp_dir,
    )


In [ ]:
TRAIN_PATH = "/kaggle/input/datasets/ai2334135131/14-model/train_filtered_14_languages.parquet" 
# TRAIN_PATH = "/kaggle/input/competitions/wsdm-cup-multilingual-chatbot-arena/train.parquet"
TEST_PATH = "/kaggle/input/competitions/wsdm-cup-multilingual-chatbot-arena/test.parquet" 
MODEL_NAME = "/kaggle/input/models/mohammedhamdan/mdeberta-v3-base/transformers/default/1/mdeberta-v3-base"
SELECTED_EXPERIMENTS = [
    #"exp1_prop_trunc_cls",
     "exp2_prop_trunc_swap_cls",
    # "exp3_prop_trunc_swap_reg",  
]

# Optional: manually import a resume checkpoint from a previous Kaggle output dataset.
# Fill these only when you want to resume from a path outside the current outputs/ directory.
# *_last.pth is required for true resume; best *.pth is optional but recommended.
MANUAL_RESUME_CHECKPOINT_PATH = ""  # e.g. "/kaggle/input/your-output/outputs/exp2_prop_trunc_swap_cls/exp2_prop_trunc_swap_cls_fold0_last.pth"
MANUAL_BEST_CHECKPOINT_PATH = ""    # e.g. "/kaggle/input/your-output/outputs/exp2_prop_trunc_swap_cls/exp2_prop_trunc_swap_cls_fold0.pth"

# For multi-fold/manual multi-experiment resume, use these dicts instead of the two single paths above.
# Example key formats: "exp2_prop_trunc_swap_cls_fold0", "fold0", "0".
MANUAL_RESUME_CHECKPOINT_PATHS = None
MANUAL_BEST_CHECKPOINT_PATHS = None

CFG = TrainConfig(
    model_name=MODEL_NAME,
    max_len=1024,  # keep full 1024 tokens; use smaller batch to fit T4x2 memory
    batch_size=2,  # global batch; DataParallel splits this across 2 GPUs => 1 per T4
    epochs=8,
    lr=2e-5,
    weight_decay=0.01,
    num_folds=2,
    seed=42,
    num_workers=0,
    output_dir="outputs",
    grad_clip=1.0,
    use_amp=True,  # mixed precision is strongly recommended on T4
    use_data_parallel=True,  # use both T4 GPUs
    head_lr=1e-4,
    gradient_accumulation_steps=8,  # keep effective batch size similar while using max_len=1024
    debug_prediction_stats=True,
    pad_to_max_length=False,
    debug_sample_size=None,
    empty_cache_after_fold=True,
    resume_training=True,
    save_last_checkpoint=True,
    skip_finished_folds=True,
    manual_resume_checkpoint_path="/kaggle/input/models/ai2334135131/model-2/transformers/default/1/exp2_prop_trunc_swap_cls_fold0_last.pth",
    manual_best_checkpoint_path=MANUAL_BEST_CHECKPOINT_PATH or None,
    manual_resume_checkpoint_paths=MANUAL_RESUME_CHECKPOINT_PATHS,
    manual_best_checkpoint_paths=MANUAL_BEST_CHECKPOINT_PATHS,
)
MAX_FOLDS_TO_RUN = 1
print(CFG)
print("Selected experiments:", SELECTED_EXPERIMENTS)
print("Available experiments:", list(EXPERIMENTS.keys()))
print("Stability config:", {"max_len": CFG.max_len, "batch_size": CFG.batch_size, "use_amp": CFG.use_amp, "use_data_parallel": CFG.use_data_parallel, "head_lr": CFG.head_lr, "gradient_accumulation_steps": CFG.gradient_accumulation_steps, "effective_batch_size": CFG.batch_size * CFG.gradient_accumulation_steps, "pad_to_max_length": CFG.pad_to_max_length, "debug_sample_size": CFG.debug_sample_size, "debug_prediction_stats": CFG.debug_prediction_stats})
print("Resume config:", {
    "resume_training": CFG.resume_training,
    "save_last_checkpoint": CFG.save_last_checkpoint,
    "skip_finished_folds": CFG.skip_finished_folds,
    "manual_resume_checkpoint_path": CFG.manual_resume_checkpoint_path,
    "manual_best_checkpoint_path": CFG.manual_best_checkpoint_path,
    "manual_resume_checkpoint_paths": CFG.manual_resume_checkpoint_paths,
    "manual_best_checkpoint_paths": CFG.manual_best_checkpoint_paths,
})


In [ ]:
all_results = {}

for exp_name in SELECTED_EXPERIMENTS:
    exp_cfg = EXPERIMENTS[exp_name]
    checkpoint_paths, oof_df, metric_df = run_cv(TRAIN_PATH, CFG, exp_cfg)
    all_results[exp_name] = {
        "checkpoint_paths": checkpoint_paths,
        "oof": oof_df,
        "metrics": metric_df,
    }

    if TEST_PATH:
        predict_test(TEST_PATH, CFG, exp_cfg, checkpoint_paths)

print("Finished experiments:", list(all_results.keys()))


In [ ]:
for exp_name, result in all_results.items():
    print(f"\n===== {exp_name} =====")
    display(result["metrics"])


In [ ]:
oof = pd.read_csv("outputs/exp1_prop_trunc_cls/exp1_prop_trunc_cls_oof.csv")
print(oof["pred_winner"].value_counts())
print(oof["prob_model_b"].describe())
